# 🔍 Detecção de Fraudes em Transações de Cartão de Crédito
### Random Forest com Boas Práticas de Data Science e MLOps

---

> **Problema:** Identificar transações fraudulentas em cartões de crédito — um problema de **classificação binária com dados severamente desbalanceados**.

> **Dataset:** `card_transdata.csv` — Features comportamentais e geográficas de transações financeiras.

---

## 📋 Estrutura do Notebook

| # | Seção | Descrição |
|---|---|---|
| 1 | **Carregamento e EDA** | Inspeção do dataset, análise de desbalanceamento e correlações |
| 2 | **Pré-processamento** | Divisão treino/teste e balanceamento com SMOTE |
| 3 | **Treinamento** | Random Forest com RandomizedSearchCV e StratifiedKFold |
| 4 | **Avaliação** | F1, ROC-AUC, Average Precision, Curvas ROC e PR |
| 5 | **Importância das Features** | Ranking de features por redução de impureza Gini |
| 6 | **MLOps** | Persistência versionada do modelo e metadados de rastreabilidade |

---

### 🏷️ Variáveis do Dataset

| Feature | Tipo | Descrição |
|---|---|---|
| `distance_from_home` | Numérico | Distância da residência até o local da transação (km) |
| `distance_from_last_transaction` | Numérico | Distância entre a última transação e a atual (km) |
| `ratio_to_median_purchase_price` | Numérico | Razão entre o valor da compra e a mediana histórica do usuário |
| `repeat_retailer` | Binário | Transação em varejista já utilizado anteriormente (1=Sim) |
| `used_chip` | Binário | Chip do cartão foi utilizado (1=Sim) |
| `used_pin_number` | Binário | PIN foi inserido (1=Sim) |
| `online_order` | Binário | Compra realizada online (1=Sim) |
| `fraud` | Binário 🎯 | **Variável alvo**: 1=Fraude, 0=Transação legítima |

In [ ]:
# ============================================================
# 📦 IMPORTS E CONFIGURAÇÕES GERAIS
# ============================================================

# Bibliotecas padrão do Python
import warnings
import os
import json
import logging
from datetime import datetime
from pathlib import Path

# Manipulação e análise de dados
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: divisão, modelos, métricas, validação
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
)

# Tratamento de desbalanceamento de classes (imbalanced-learn)
from imblearn.over_sampling import SMOTE

# Persistência eficiente do modelo (MLOps)
import joblib

# ── Configurações globais ────────────────────────────────────────────────────

warnings.filterwarnings("ignore")

# Estilo de visualização consistente
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100})
sns.set_theme(style="whitegrid", palette="husl")

# Logging estruturado — fundamental em MLOps para rastreabilidade de execuções
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("fraud_detection")

# ── Semente aleatória — garante reprodutibilidade dos experimentos ───────────
# Boa prática de MLOps: fixar sempre o random_state em todo o pipeline
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Caminhos do projeto ──────────────────────────────────────────────────────
DATA_PATH = Path("data/input/card_transdata.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)  # Cria o diretório de modelos se não existir

logger.info("✅ Ambiente configurado com sucesso.")
print(f"  Python working directory: {Path.cwd()}")
print(f"  Model output dir:         {MODEL_DIR.resolve()}")

---

## 1. Carregamento e Exploração dos Dados (EDA)

> **Objetivo:** Entender a estrutura do dataset, detectar problemas de qualidade e identificar padrões relevantes **antes** de qualquer modelagem.

### Por que a EDA é obrigatória?

- Revela desbalanceamento de classes — crítico para fraude
- Identifica outliers que podem distorcer o modelo
- Expõe correlações que orientam a seleção de features
- Detecta vazamento de dados (*data leakage*) precocemente

In [ ]:
def load_and_inspect_data(filepath: Path) -> pd.DataFrame:
    """
    Carrega o dataset CSV e exibe um diagnóstico completo da estrutura dos dados.

    Esta etapa inicial é fundamental para:
    - Confirmar que os dados foram lidos corretamente
    - Identificar tipos de dados inconsistentes
    - Detectar valores ausentes antes de qualquer transformação
    - Obter visão geral das distribuições com estatísticas descritivas

    Args:
        filepath: Caminho para o arquivo CSV.

    Returns:
        pd.DataFrame: DataFrame com os dados carregados e prontos para análise.
    """
    df = pd.read_csv(filepath)
    logger.info(f"Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

    print("=" * 65)
    print("📐 DIMENSÕES DO DATASET")
    print(f"   Linhas  : {df.shape[0]:,}")
    print(f"   Colunas : {df.shape[1]}")

    print("\n📊 TIPOS DE DADOS POR COLUNA")
    for col, dtype in df.dtypes.items():
        print(f"   {col:<40} {dtype}")

    print("\n🔍 VALORES AUSENTES")
    missing = df.isnull().sum()
    if missing.any():
        print(missing[missing > 0].to_string())
    else:
        print("   Nenhum valor ausente encontrado ✅")

    print("\n📈 ESTATÍSTICAS DESCRITIVAS")
    print(df.describe().T.round(4).to_string())
    print("=" * 65)

    return df


# Carrega os dados e inspeciona a estrutura
df = load_and_inspect_data(DATA_PATH)

# Exibe as primeiras linhas para inspeção visual
print("\n🔎 Primeiras 5 linhas do dataset:")
df.head()

In [ ]:
def plot_class_distribution(df: pd.DataFrame, target_col: str = "fraud") -> None:
    """
    Visualiza a distribuição da variável alvo e calcula métricas de desbalanceamento.

    Em detecção de fraude, o desbalanceamento é esperado e severo.
    Modelos que ignoram isso atingem alta acurácia prevendo sempre "legítima",
    mas têm recall próximo de zero para fraudes — inúteis na prática.

    O *imbalance ratio* (razão entre classes) orienta a escolha da estratégia:
    - < 10:1  → class_weight='balanced' geralmente suficiente
    - > 10:1  → SMOTE ou outras técnicas de reamostragem recomendadas

    Args:
        df: DataFrame com os dados.
        target_col: Nome da coluna alvo (default: "fraud").
    """
    counts = df[target_col].value_counts().sort_index()
    pct = df[target_col].value_counts(normalize=True).sort_index() * 100
    labels = {0: "Legítima", 1: "Fraude"}
    colors = ["#2ecc71", "#e74c3c"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfico de barras com contagem absoluta
    bars = axes[0].bar(
        [labels[k] for k in counts.index],
        counts.values,
        color=colors,
        edgecolor="white",
        linewidth=1.5,
        width=0.5,
    )
    for bar, val in zip(bars, counts.values):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f"{val:,}",
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=11,
        )
    axes[0].set_title("Distribuição Absoluta das Classes", fontsize=13)
    axes[0].set_ylabel("Número de Transações")
    axes[0].set_ylim(0, counts.max() * 1.15)

    # Gráfico de pizza com proporção percentual
    axes[1].pie(
        pct.values,
        labels=[f"{labels[k]}\n{v:.2f}%" for k, v in zip(pct.index, pct.values)],
        colors=colors,
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2},
        textprops={"fontsize": 12},
    )
    axes[1].set_title("Proporção das Classes", fontsize=13)

    plt.suptitle(
        "⚠️ Desbalanceamento de Classes — Problema Típico de Detecção de Fraude",
        fontsize=14,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # Métricas de desbalanceamento
    imbalance_ratio = counts[0] / counts[1]
    print(f"\n📊 Classe 0 (Legítima): {counts[0]:,}  ({pct[0]:.2f}%)")
    print(f"📊 Classe 1 (Fraude):   {counts[1]:,}  ({pct[1]:.2f}%)")
    print(f"⚖️  Razão de desbalanceamento: {imbalance_ratio:.1f}:1")
    print(
        f"\n💡 Acurácia de um classificador 'dummy' (sempre prevê 0): {pct[0]:.2f}%"
        " → métrica enganosa!"
    )


def plot_feature_distributions(df: pd.DataFrame, target_col: str = "fraud") -> None:
    """
    Visualiza a distribuição de cada feature separada por classe (Legítima vs Fraude).

    Features com distribuições claramente distintas entre as classes têm alto
    poder discriminativo — são as mais valiosas para o modelo.

    Histogramas normalizados (density=True) permitem comparação direta mesmo
    com classes desbalanceadas.

    Args:
        df: DataFrame com os dados.
        target_col: Nome da coluna alvo (default: "fraud").
    """
    features = [c for c in df.columns if c != target_col]
    n_cols = 3
    n_rows = (len(features) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten()

    colors = {0: "#2ecc71", 1: "#e74c3c"}
    labels = {0: "Legítima", 1: "Fraude"}

    for i, feat in enumerate(features):
        for cls in [0, 1]:
            subset = df[df[target_col] == cls][feat]
            axes[i].hist(
                subset,
                bins=40,
                alpha=0.65,
                label=labels[cls],
                color=colors[cls],
                density=True,  # Normaliza para comparar classes desbalanceadas
            )
        axes[i].set_title(feat, fontsize=11)
        axes[i].legend(fontsize=9)
        axes[i].set_ylabel("Densidade")
        axes[i].set_xlabel(feat)

    # Remove eixos extras se o número de features não preenche a grade
    for j in range(len(features), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(
        "Distribuição das Features por Classe (normalizada)",
        fontsize=14,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


def plot_correlation_heatmap(df: pd.DataFrame) -> None:
    """
    Exibe a matriz de correlação de Pearson entre todas as variáveis numéricas.

    Interprete os valores como:
    - |r| > 0.7 → correlação forte (potencial multicolinearidade entre features)
    - |r| alto com 'fraud' → feature relevante para o modelo

    Nota: Correlação de Pearson mede relações *lineares*. Random Forest captura
    relações não-lineares, portanto features com baixa correlação linear
    podem ainda ser informativas.

    Args:
        df: DataFrame com variáveis numéricas.
    """
    corr = df.corr()
    # Máscara triangular superior para evitar redundância visual
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        vmin=-1,
        vmax=1,
        ax=ax,
        linewidths=0.5,
        square=True,
        annot_kws={"size": 9},
    )
    ax.set_title("Matriz de Correlação de Pearson", fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


# Executa as três análises visuais
plot_class_distribution(df)
plot_feature_distributions(df)
plot_correlation_heatmap(df)

---

## 2. Pré-processamento e Tratamento de Desbalanceamento

> **Problema central:** Com ~91% de transações legítimas, um modelo ingênuo que sempre prevê "sem fraude" teria 91% de acurácia — mas **0% de utilidade prática**.

### Estratégias para dados desbalanceados (da mais simples à mais robusta)

| Estratégia | Como funciona | Quando usar |
|---|---|---|
| `class_weight='balanced'` | Pondera classes inversamente à frequência | Desbalanceamento moderado, baseline rápido |
| **SMOTE** | Gera amostras sintéticas da classe minoritária por interpolação entre k-vizinhos | Desbalanceamento severo (> 10:1) |
| Undersampling | Remove amostras da classe majoritária | Datasets muito grandes, risco de perda de info |
| Combinado | SMOTE + Undersampling (SMOTEENN, SMOTETomek) | Casos extremos |

### ⚠️ Regra de ouro: SMOTE apenas no treino!

O SMOTE **deve ser aplicado exclusivamente no conjunto de treino** para evitar *data leakage*:
- Aplicar antes do split → amostras sintéticas contaminam o teste
- Aplicar após o split no treino → avaliação honesta e resultados generalizáveis

In [ ]:
# ── Definição das features e do target ──────────────────────────────────────
TARGET = "fraud"
FEATURES = [col for col in df.columns if col != TARGET]

X = df[FEATURES]
y = df[TARGET]

logger.info(f"Target    : {TARGET}")
logger.info(f"Features  : {FEATURES}")
logger.info(f"Shape X   : {X.shape}  |  Shape y: {y.shape}")

# ── Divisão treino/teste com estratificação ──────────────────────────────────
# stratify=y: ESSENCIAL para dados desbalanceados
# Garante que a proporção de fraudes seja preservada em treino e teste,
# evitando que um split aleatório concentre todas as fraudes em um único conjunto.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,          # 80% treino / 20% teste
    random_state=RANDOM_STATE,
    stratify=y,              # Preserva a proporção de classes
)

logger.info(f"Treino : {X_train.shape[0]:,} amostras")
logger.info(f"Teste  : {X_test.shape[0]:,} amostras")

# Verifica que a proporção foi mantida em ambos os conjuntos
print("\n📊 Distribuição da classe no conjunto de TREINO:")
train_dist = y_train.value_counts(normalize=True).sort_index() * 100
for cls, pct in train_dist.items():
    label = "Legítima" if cls == 0 else "Fraude  "
    print(f"  Classe {cls} ({label}): {pct:.2f}%")

print("\n📊 Distribuição da classe no conjunto de TESTE:")
test_dist = y_test.value_counts(normalize=True).sort_index() * 100
for cls, pct in test_dist.items():
    label = "Legítima" if cls == 0 else "Fraude  "
    print(f"  Classe {cls} ({label}): {pct:.2f}%")

In [ ]:
def apply_smote(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    random_state: int = 42,
    k_neighbors: int = 5,
) -> tuple:
    """
    Aplica SMOTE (Synthetic Minority Oversampling Technique) exclusivamente
    no conjunto de treino para balancear as classes.

    Como o SMOTE funciona:
    1. Para cada amostra da classe minoritária (fraude), encontra os k vizinhos
       mais próximos no espaço de features.
    2. Gera novos pontos sintéticos ao longo dos segmentos de reta entre a
       amostra original e seus vizinhos — interpolação linear.
    3. Repete até atingir o balanceamento desejado (padrão: 1:1).

    Vantagem sobre oversampling simples (duplicação): os pontos sintéticos
    introduzem variabilidade, reduzindo overfitting na classe minoritária.

    ⚠️ NUNCA aplique SMOTE antes de dividir treino/teste:
       Vazamento de dados (data leakage) tornaria a avaliação otimista e irreal.

    Args:
        X_train: Features do conjunto de treino.
        y_train: Labels do conjunto de treino.
        random_state: Semente para reprodutibilidade.
        k_neighbors: Número de vizinhos para interpolação (padrão: 5).

    Returns:
        Tupla (X_resampled, y_resampled) com dados balanceados.
    """
    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    X_res, y_res = smote.fit_resample(X_train, y_train)

    before = pd.Series(y_train).value_counts().sort_index()
    after = pd.Series(y_res).value_counts().sort_index()

    print("📊 Distribuição ANTES do SMOTE (treino original):")
    print(f"  Legítima (0): {before[0]:,}  |  Fraude (1): {before[1]:,}")
    print(f"  Razão: {before[0] / before[1]:.1f}:1")

    print("\n📊 Distribuição APÓS o SMOTE (treino balanceado):")
    print(f"  Legítima (0): {after[0]:,}  |  Fraude (1): {after[1]:,}")
    print(f"  Razão: {after[0] / after[1]:.1f}:1")

    print(f"\n📈 Amostras de treino: {X_train.shape[0]:,} → {X_res.shape[0]:,}")
    print(f"   Amostras sintéticas criadas: {X_res.shape[0] - X_train.shape[0]:,}")

    logger.info(
        f"SMOTE aplicado: {X_train.shape[0]:,} → {X_res.shape[0]:,} amostras de treino"
    )
    return X_res, y_res


# Aplica SMOTE somente no treino — teste permanece intocado para avaliação honesta
X_train_res, y_train_res = apply_smote(X_train, y_train, RANDOM_STATE)

---

## 3. Treinamento do Modelo — Random Forest

### Por que Random Forest para detecção de fraude?

O **Random Forest** é um método de ensemble que combina múltiplas árvores de decisão independentes:

- **Bagging:** cada árvore é treinada em uma amostra *bootstrap* (amostragem com reposição) do dataset
- **Feature randomness:** em cada split, apenas um subconjunto aleatório de features é avaliado (`max_features='sqrt'`)
- **Votação:** a predição final é a votação majoritária (classificação) de todas as árvores

### Vantagens para este problema

| Característica | Benefício para Fraude |
|---|---|
| Robusto a outliers | Transações fraudulentas frequentemente têm valores extremos |
| Importância nativa de features | Explica quais variáveis mais contribuem para a decisão |
| Não requer normalização | Features em escalas diferentes não afetam o resultado |
| Regularização embutida | `min_samples_leaf` e `max_depth` controlam overfitting |
| Paralelizável (`n_jobs=-1`) | Treinamento eficiente em múltiplos núcleos |

### Otimização de Hiperparâmetros

Usamos **`RandomizedSearchCV`** em vez de `GridSearchCV` porque:
- O espaço de busca é grande → busca exaustiva seria impraticável
- Amostragem aleatória de combinações frequentemente encontra soluções igualmente boas em fração do tempo
- Combinado com `StratifiedKFold(5)` para validação cruzada robusta
- Métrica de otimização: **F1-score** (penaliza tanto falsos positivos quanto falsos negativos)

In [ ]:
def build_param_grid() -> dict:
    """
    Define o espaço de busca de hiperparâmetros para o Random Forest.

    Descrição de cada hiperparâmetro e seu impacto:

    - n_estimators: Número de árvores.
      Mais árvores → menor variância, melhor generalização, custo computacional maior.
      Diminishing returns acima de ~300-500.

    - max_depth: Profundidade máxima de cada árvore.
      None → árvore cresce até pureza total (risco de overfitting).
      Limitar a profundidade regulariza o modelo.

    - min_samples_split: Mínimo de amostras para dividir um nó interno.
      Valores maiores → árvores mais rasas → regularização.

    - min_samples_leaf: Mínimo de amostras em nós folha.
      Aumentar evita folhas com apenas 1 amostra (overfitting a ruído).

    - max_features: Número de features avaliadas em cada split.
      'sqrt' = sqrt(n_features) → padrão recomendado para classificação.

    - class_weight: Pesos das classes para compensar desbalanceamento.
      'balanced_subsample' re-calcula os pesos a cada bootstrap sample.

    Returns:
        dict: Espaço de hiperparâmetros para RandomizedSearchCV.
    """
    return {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
        "class_weight": ["balanced", "balanced_subsample"],
    }


def train_random_forest(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    param_grid: dict,
    random_state: int = 42,
    n_iter: int = 30,
) -> tuple:
    """
    Treina um Random Forest com busca aleatória de hiperparâmetros
    usando validação cruzada estratificada.

    Fluxo:
    1. Define RandomForestClassifier base
    2. Configura StratifiedKFold para preservar proporção de classes em cada fold
    3. Executa RandomizedSearchCV testando n_iter combinações aleatórias
    4. Retreina o melhor modelo em todo o conjunto de treino (refit=True)

    Args:
        X_train: Features de treino (após SMOTE).
        y_train: Labels de treino (após SMOTE).
        param_grid: Espaço de hiperparâmetros.
        random_state: Semente para reprodutibilidade.
        n_iter: Número de combinações aleatórias a testar.

    Returns:
        Tupla (melhor_modelo, objeto_cv_search) com modelo otimizado e metadados da busca.
    """
    rf = RandomForestClassifier(random_state=random_state, n_jobs=-1)

    # StratifiedKFold preserva a proporção das classes em cada fold de validação
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    # RandomizedSearchCV: mais eficiente que GridSearchCV em espaços grandes
    # scoring='f1' otimiza para o equilíbrio entre precision e recall — ideal para fraude
    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring="f1",           # Métrica de otimização: F1 da classe positiva (fraude)
        cv=cv_strategy,
        refit=True,             # Retreina com melhores params em todo X_train após a busca
        random_state=random_state,
        n_jobs=-1,
        verbose=1,
    )

    logger.info(f"Iniciando busca de hiperparâmetros ({n_iter} iterações × 5 folds)...")
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    logger.info(f"✅ Melhor F1-score (CV treino): {search.best_score_:.4f}")

    print("\n" + "=" * 55)
    print("🏆 MELHORES HIPERPARÂMETROS ENCONTRADOS:")
    print("=" * 55)
    for param, value in search.best_params_.items():
        print(f"  {param:<25} → {value}")
    print(f"\n  F1-score médio (CV, 5 folds): {search.best_score_:.4f}")
    print("=" * 55)

    return best_model, search


# ── Executa o treinamento ────────────────────────────────────────────────────
param_grid = build_param_grid()
best_rf, cv_search = train_random_forest(
    X_train_res,
    y_train_res,
    param_grid,
    random_state=RANDOM_STATE,
    n_iter=30,
)

In [ ]:
def evaluate_with_cross_validation(
    model: RandomForestClassifier,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Avalia o modelo final com validação cruzada estratificada usando múltiplas métricas.

    A validação cruzada fornece uma estimativa mais robusta do desempenho que uma única
    divisão treino/validação, pois:
    - Todos os dados são usados para validação (em rodadas diferentes)
    - Reduz a variância da estimativa de desempenho
    - Detecta overfitting ao revelar alta variância entre os folds

    Métricas calculadas:
    - f1         : F1-score da classe positiva (fraude) — métrica principal
    - f1_macro   : Média dos F1 de ambas as classes — visão equilibrada
    - precision  : Precisão na classe positiva — taxa de acerto dos alertas
    - recall     : Sensibilidade — taxa de detecção de fraudes reais
    - roc_auc    : Área sob a curva ROC — capacidade discriminativa geral

    Args:
        model: RandomForestClassifier com os melhores hiperparâmetros.
        X_train: Features de treino (após SMOTE).
        y_train: Labels de treino (após SMOTE).
        random_state: Semente aleatória.

    Returns:
        pd.DataFrame: Métricas por fold (5 linhas × 5 colunas de métricas).
    """
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    scoring = {
        "f1"        : "f1",
        "f1_macro"  : "f1_macro",
        "precision" : "precision",
        "recall"    : "recall",
        "roc_auc"   : "roc_auc",
    }

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=-1,
    )

    # Organiza resultados em DataFrame legível
    metrics_df = pd.DataFrame(
        {k.replace("test_", ""): v for k, v in results.items() if k.startswith("test_")}
    )

    summary = metrics_df.agg(["mean", "std"]).T
    summary.columns = ["Média", "Desvio Padrão"]
    summary["IC 95% Lower"] = summary["Média"] - 1.96 * summary["Desvio Padrão"]
    summary["IC 95% Upper"] = summary["Média"] + 1.96 * summary["Desvio Padrão"]

    print("=" * 65)
    print("📊 VALIDAÇÃO CRUZADA — 5-Fold Estratificado (Melhor Modelo)")
    print("=" * 65)
    print(summary.round(4).to_string())
    print("=" * 65)
    print("\n💡 IC 95% baseado na distribuição dos folds (±1.96 × desvio padrão)")

    return metrics_df


cv_results = evaluate_with_cross_validation(best_rf, X_train_res, y_train_res, RANDOM_STATE)

---

## 4. Avaliação do Modelo no Conjunto de Teste

> **Por que não usar acurácia?**
>
> Com 91.3% de transações legítimas, um modelo que sempre prevê "legítima" atinge **91.3% de acurácia** mas tem **recall = 0%** para fraudes.
> Usar acurácia em dados desbalanceados é uma armadilha clássica!

### Métricas adequadas para detecção de fraude

| Métrica | Fórmula | Interpretação no contexto de fraude |
|---|---|---|
| **Precision** | $\frac{TP}{TP+FP}$ | Dos alertas de fraude gerados, quantos são realmente fraude? |
| **Recall (Sensibilidade)** | $\frac{TP}{TP+FN}$ | Das fraudes reais, quantas foram detectadas? |
| **F1-Score** | $\frac{2 \cdot P \cdot R}{P+R}$ | Balanço harmônico entre Precision e Recall |
| **ROC-AUC** | Área sob ROC | Capacidade geral de discriminação (1.0 = perfeito) |
| **Average Precision** | Área sob P-R | Mais informativo que ROC-AUC para dados desbalanceados |

### Trade-off Precision × Recall

Em fraude financeira, geralmente preferimos **maximizar Recall**:
- **Falso Negativo** (fraude não detectada) → **custo alto** para o cliente e para o banco
- **Falso Positivo** (alerta indevido) → **custo baixo** (bloqueio temporário, inconveniência)

O threshold de decisão pode ser ajustado para controlar esse trade-off.

In [ ]:
def evaluate_model_on_test(
    model: RandomForestClassifier,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict:
    """
    Avalia o modelo treinado no conjunto de teste com um conjunto completo de métricas.

    O conjunto de teste jamais deve ser visto durante o treinamento ou tunning —
    representa dados completamente novos e fornece uma estimativa honesta do
    desempenho do modelo em produção.

    Args:
        model: RandomForestClassifier treinado.
        X_test: Features do conjunto de teste (dados originais, sem SMOTE).
        y_test: Labels verdadeiros do conjunto de teste.

    Returns:
        dict: Dicionário com métricas escalares e vetores de predição/probabilidade.
    """
    # Predição binária (threshold padrão = 0.5)
    y_pred = model.predict(X_test)
    # Probabilidade estimada para a classe positiva (fraude)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc    = roc_auc_score(y_test, y_proba)
    avg_prec   = average_precision_score(y_test, y_proba)
    f1         = f1_score(y_test, y_pred)

    print("=" * 65)
    print("📋 RELATÓRIO DE CLASSIFICAÇÃO — CONJUNTO DE TESTE")
    print("=" * 65)
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Legítima (0)", "Fraude (1)"],
            digits=4,
        )
    )
    print(f"  🎯 ROC-AUC Score             : {roc_auc:.4f}")
    print(f"  🎯 Average Precision (PR-AUC): {avg_prec:.4f}")
    print("=" * 65)

    # Interpretação rápida
    print("\n📌 Interpretação:")
    print(f"  • A cada 100 alertas de fraude, ~{int(f1 * 100)} são aproveitáveis (F1 proxy)")
    print(f"  • O modelo discrimina fraudes com {roc_auc*100:.1f}% de confiança (ROC-AUC)")

    return {
        "roc_auc"           : roc_auc,
        "average_precision" : avg_prec,
        "f1_score"          : f1,
        "y_pred"            : y_pred,
        "y_proba"           : y_proba,
    }


test_metrics = evaluate_model_on_test(best_rf, X_test, y_test)

In [ ]:
def plot_evaluation_charts(
    y_test: pd.Series,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> None:
    """
    Gera três gráficos complementares de avaliação:

    1. **Matriz de Confusão**: Decompõe predições em TP, TN, FP, FN.
       Essencial para entender os tipos de erros do modelo.

    2. **Curva ROC** (Receiver Operating Characteristic):
       Trade-off entre Taxa de Verdadeiros Positivos (TPR) e Taxa de Falsos
       Positivos (FPR) em todos os thresholds possíveis.
       AUC = 0.5 → aleatório | AUC = 1.0 → perfeito.

    3. **Curva Precision-Recall**:
       Mais informativa que ROC-AUC para dados desbalanceados.
       Foca no desempenho na classe minoritária (fraude).
       Baseline = proporção de positivos no dataset.

    Args:
        y_test: Labels verdadeiros do conjunto de teste.
        y_pred: Predições binárias do modelo.
        y_proba: Probabilidades para a classe positiva (fraude).
    """
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # ── 1. Matriz de Confusão ────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legítima", "Fraude"])
    disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Matriz de Confusão", fontsize=13, pad=10)

    tn, fp, fn, tp = cm.ravel()
    axes[0].set_xlabel(
        f"Verdadeiro Negativo (TN): {tn:,}\n"
        f"Falso Positivo (FP):      {fp:,}\n"
        f"Falso Negativo (FN):      {fn:,}  ← fraudes perdidas\n"
        f"Verdadeiro Positivo (TP): {tp:,}",
        fontsize=9,
    )

    # ── 2. Curva ROC ─────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    axes[1].plot(fpr, tpr, color="#e74c3c", lw=2.5, label=f"Random Forest (AUC = {roc_auc:.4f})")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.6, label="Classificador aleatório (AUC = 0.50)")
    axes[1].fill_between(fpr, tpr, alpha=0.08, color="#e74c3c")
    axes[1].set_xlabel("Taxa de Falsos Positivos (FPR)", fontsize=11)
    axes[1].set_ylabel("Taxa de Verdadeiros Positivos (TPR)", fontsize=11)
    axes[1].set_title("Curva ROC", fontsize=13, pad=10)
    axes[1].legend(loc="lower right", fontsize=9)
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1.02])

    # ── 3. Curva Precision-Recall ────────────────────────────────────────────
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
    avg_prec = average_precision_score(y_test, y_proba)
    baseline = y_test.mean()  # Proporção de fraudes — baseline de um modelo aleatório

    axes[2].plot(recall_vals, precision_vals, color="#3498db", lw=2.5, label=f"Random Forest (AP = {avg_prec:.4f})")
    axes[2].axhline(
        baseline,
        color="gray",
        linestyle="--",
        lw=1.5,
        alpha=0.7,
        label=f"Baseline aleatório ({baseline:.3f})",
    )
    axes[2].fill_between(recall_vals, precision_vals, alpha=0.08, color="#3498db")
    axes[2].set_xlabel("Recall (Sensibilidade)", fontsize=11)
    axes[2].set_ylabel("Precision", fontsize=11)
    axes[2].set_title("Curva Precision-Recall", fontsize=13, pad=10)
    axes[2].legend(loc="upper right", fontsize=9)
    axes[2].set_xlim([0, 1])
    axes[2].set_ylim([0, 1.02])

    plt.suptitle(
        "Avaliação Completa do Modelo — Random Forest (Conjunto de Teste)",
        fontsize=14,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


plot_evaluation_charts(y_test, test_metrics["y_pred"], test_metrics["y_proba"])

---

## 5. Importância das Features

> A importância das features no Random Forest é calculada como a **redução média de impureza de Gini** trazida por cada feature em todas as árvores do ensemble, ponderada pelo número de amostras que passam por cada nó.

### Interpretação

- Features com **maior importância** são as mais decisivas para distinguir fraude de transação legítima
- O **desvio padrão** entre as árvores indica a estabilidade da importância — alta variância sugere instabilidade
- Features com importância próxima de zero podem ser candidatas à remoção (simplificação do modelo)

### Limitação conhecida

A importância por impureza de Gini pode **superestimar features contínuas** e com alta cardinalidade. Para interpretabilidade mais robusta, considere `permutation_importance` em uma análise complementar.

In [ ]:
def plot_feature_importance(
    model: RandomForestClassifier,
    feature_names: list,
) -> pd.DataFrame:
    """
    Visualiza e retorna a importância das features do Random Forest.

    A importância por impureza de Gini é calculada como:
        importance(feature_j) = Σ [p(nó) × ΔGini(nó)] para todos os nós
        que usam feature_j, média sobre todas as árvores do ensemble.

    Onde p(nó) = fração de amostras que passam pelo nó.

    O desvio padrão entre árvores (barras de erro) indica a estabilidade:
    - Desvio baixo → feature consistentemente importante em todas as árvores
    - Desvio alto  → importância varia bastante entre árvores (instabilidade)

    Args:
        model: RandomForestClassifier treinado.
        feature_names: Lista com os nomes das features na mesma ordem do treino.

    Returns:
        pd.DataFrame: Ranking de features por importância decrescente.
    """
    importances = model.feature_importances_
    # Calcula o desvio padrão da importância entre as árvores individuais
    std_devs = np.std(
        [tree.feature_importances_ for tree in model.estimators_], axis=0
    )

    feat_imp_df = (
        pd.DataFrame(
            {
                "feature"    : feature_names,
                "importance" : importances,
                "std"        : std_devs,
            }
        )
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    # Importância relativa percentual para facilitar a interpretação
    feat_imp_df["importance_pct"] = (
        feat_imp_df["importance"] / feat_imp_df["importance"].sum() * 100
    )

    # ── Gráfico de barras horizontais com barras de erro ──────────────────
    fig, ax = plt.subplots(figsize=(11, 6))
    palette = sns.color_palette("husl", len(feature_names))

    bars = ax.barh(
        feat_imp_df["feature"][::-1],
        feat_imp_df["importance"][::-1],
        xerr=feat_imp_df["std"][::-1],
        color=palette,
        edgecolor="white",
        linewidth=1.2,
        error_kw={"elinewidth": 1.8, "ecolor": "#2c3e50", "capsize": 5},
    )

    # Anotações com a porcentagem de importância relativa
    for bar, pct in zip(bars, feat_imp_df["importance_pct"][::-1]):
        ax.text(
            bar.get_width() + 0.003,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%",
            va="center",
            fontsize=10,
            fontweight="bold",
        )

    ax.set_xlabel("Importância Média (redução de impureza Gini)", fontsize=11)
    ax.set_title(
        "Importância das Features — Random Forest\n(média ± desvio padrão entre árvores)",
        fontsize=13,
    )
    ax.set_xlim(0, feat_imp_df["importance"].max() * 1.28)
    plt.tight_layout()
    plt.show()

    print("\n🔍 RANKING COMPLETO DE IMPORTÂNCIA DAS FEATURES:")
    print("=" * 60)
    print(
        feat_imp_df[["feature", "importance", "std", "importance_pct"]]
        .rename(columns={
            "importance"    : "Importância",
            "std"           : "Desvio Padrão",
            "importance_pct": "Importância (%)",
        })
        .to_string(index=False, float_format=lambda x: f"{x:.5f}")
    )

    return feat_imp_df


feat_importance_df = plot_feature_importance(best_rf, FEATURES)

---

## 6. MLOps — Persistência, Rastreabilidade e Inferência

> **MLOps** (Machine Learning Operations) é o conjunto de práticas e ferramentas que tornam os modelos de ML confiáveis, reprodutíveis e operacionalizáveis em produção.

### Boas práticas implementadas neste projeto

| Prática MLOps | Implementação |
|---|---|
| **Reprodutibilidade** | `RANDOM_STATE = 42` fixo em todo o pipeline |
| **Versionamento** | Timestamp no nome do arquivo do modelo |
| **Rastreabilidade** | Metadados em JSON: hiperparâmetros, métricas, features, estratégias |
| **Serialização eficiente** | `joblib` com compressão (menor que `pickle` para arrays NumPy) |
| **Anti-leakage** | SMOTE aplicado apenas no treino; teste permanece virgem |
| **Separação de artefatos** | Modelo `.joblib` + metadados `.json` no diretório `models/` |
| **Logging estruturado** | Formato padronizado com timestamp para monitoramento |

### Próximos passos naturais (roadmap MLOps)

1. 🔄 **MLflow / DVC** — versionamento de experimentos e datasets
2. 🐳 **Docker** — empacotamento do ambiente de inferência
3. ⚡ **FastAPI / BentoML** — API REST para inferência em tempo real
4. 📊 **Evidently / Prometheus** — monitoramento de drift de dados em produção
5. 🔁 **CI/CD** — retreinamento automático com novos dados

In [ ]:
def save_model_artifacts(
    model: RandomForestClassifier,
    metrics: dict,
    features: list,
    param_search: RandomizedSearchCV,
    model_dir: Path,
    random_state: int,
) -> str:
    """
    Persiste o modelo treinado e seus metadados de rastreabilidade.

    Artefatos gerados:
    - `random_forest_fraud_<timestamp>.joblib`: modelo serializado com joblib
      (compressão nível 3 — bom equilíbrio entre tamanho e velocidade de I/O)
    - `metadata_<timestamp>.json`: ficha técnica do modelo com hiperparâmetros,
      métricas de avaliação, features e configurações de MLOps

    O timestamp no nome garante que múltiplas execuções não sobrescrevam
    artefatos anteriores — estratégia de versionamento simples e eficaz.

    Args:
        model: RandomForestClassifier treinado e otimizado.
        metrics: Dicionário de métricas do conjunto de teste.
        features: Lista de features utilizadas no treino.
        param_search: Objeto RandomizedSearchCV com histórico da busca.
        model_dir: Diretório de destino para os artefatos.
        random_state: Semente aleatória utilizada em todo o pipeline.

    Returns:
        str: Caminho absoluto do arquivo do modelo salvo.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_filename   = f"random_forest_fraud_{timestamp}.joblib"
    metadata_filename = f"metadata_{timestamp}.json"
    model_path    = model_dir / model_filename
    metadata_path = model_dir / metadata_filename

    # ── Serializa o modelo com joblib (eficiente para arrays NumPy) ──────────
    joblib.dump(model, model_path, compress=3)
    logger.info(f"Modelo serializado em: {model_path}")

    # ── Constrói metadados estruturados ─────────────────────────────────────
    metadata = {
        "model_info": {
            "name"        : "RandomForestClassifier",
            "version"     : timestamp,
            "saved_at"    : datetime.now().isoformat(),
            "random_state": random_state,
            "file"        : model_filename,
        },
        "training": {
            "best_params"          : param_search.best_params_,
            "cv_best_f1_score"     : round(float(param_search.best_score_), 4),
            "cv_strategy"          : "StratifiedKFold(n_splits=5)",
            "hyperparameter_method": "RandomizedSearchCV(n_iter=30)",
            "features"             : features,
            "n_features"           : len(features),
            "target"               : "fraud",
            "resampling_strategy"  : "SMOTE(k_neighbors=5) — apenas no treino",
            "train_size"           : int(X_train_res.shape[0]),
            "test_size"            : int(X_test.shape[0]),
        },
        "evaluation_test_set": {
            "roc_auc"           : round(float(metrics["roc_auc"]), 4),
            "average_precision" : round(float(metrics["average_precision"]), 4),
            "f1_score"          : round(float(metrics["f1_score"]), 4),
        },
        "mlops": {
            "framework"            : "scikit-learn",
            "imbalanced_lib"       : "imbalanced-learn (SMOTE)",
            "serialization"        : "joblib (compress=3)",
            "reproducibility_seed" : random_state,
        },
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    logger.info(f"Metadados salvos em: {metadata_path}")

    print("\n✅ ARTEFATOS DO MODELO SALVOS COM SUCESSO:")
    print(f"   📦 Modelo    : {model_path.resolve()}")
    print(f"   📄 Metadados : {metadata_path.resolve()}")
    print("\n📋 FICHA TÉCNICA DO MODELO:")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))

    return str(model_path.resolve())


# Salva modelo e metadados
model_path = save_model_artifacts(
    model=best_rf,
    metrics=test_metrics,
    features=FEATURES,
    param_search=cv_search,
    model_dir=MODEL_DIR,
    random_state=RANDOM_STATE,
)

In [ ]:
def load_model_and_predict(
    model_path: str,
    new_data: pd.DataFrame,
    fraud_threshold: float = 0.5,
) -> pd.DataFrame:
    """
    Carrega um modelo persistido e realiza inferência em novos dados.

    Demonstra o ciclo completo de MLOps:
        Treinar → Avaliar → Persistir → Carregar → Inferir

    O threshold de decisão (padrão=0.5) pode ser ajustado para controlar
    o trade-off Precision × Recall:
    - Threshold menor → mais alertas de fraude (maior recall, menor precision)
    - Threshold maior → menos alertas (menor recall, maior precision)

    Em produção, o threshold ideal é calibrado com base no custo de cada
    tipo de erro (falso positivo vs. falso negativo).

    Args:
        model_path: Caminho para o arquivo .joblib do modelo.
        new_data: DataFrame com as mesmas features usadas no treino.
        fraud_threshold: Limiar de probabilidade para classificar como fraude.

    Returns:
        pd.DataFrame: Dados originais enriquecidos com probabilidade e predição.
    """
    # Carrega o modelo serializado
    loaded_model = joblib.load(model_path)
    logger.info(f"Modelo carregado de: {model_path}")

    # Garante que as features estão na ordem correta
    new_data = new_data[FEATURES]

    proba      = loaded_model.predict_proba(new_data)[:, 1]
    prediction = (proba >= fraud_threshold).astype(int)

    result_df = new_data.copy()
    result_df["prob_fraude"]  = proba.round(4)
    result_df["pred_fraude"]  = prediction
    result_df["alerta"]       = result_df["pred_fraude"].map({0: "✅ Legítima", 1: "🚨 Fraude"})

    return result_df


# ── Demonstração de inferência com o modelo carregado ───────────────────────
print("=" * 65)
print("🔄 DEMONSTRAÇÃO: Ciclo completo de inferência com modelo em disco")
print("=" * 65)

# Usa as 10 primeiras amostras do teste como "novos dados de produção"
sample_data     = X_test.iloc[:10].copy()
sample_labels   = y_test.iloc[:10].values

result = load_model_and_predict(model_path, sample_data, fraud_threshold=0.5)
result["real"] = ["🚨 Fraude" if l == 1 else "✅ Legítima" for l in sample_labels]

print("\n📋 Predições vs. Rótulos Reais (10 primeiras amostras do teste):")
print(result[["prob_fraude", "pred_fraude", "alerta", "real"]].to_string())

# Resumo final do projeto
print("\n" + "=" * 65)
print("🏁 RESUMO DO PIPELINE COMPLETO")
print("=" * 65)
etapas = [
    ("1. EDA",              "Distribuição, correlações e análise de desbalanceamento"),
    ("2. Pré-processamento", "Split estratificado (80/20) + SMOTE no treino"),
    ("3. Modelagem",         "Random Forest + RandomizedSearchCV (30 iter × 5 folds)"),
    ("4. Avaliação",         "F1, ROC-AUC, Average Precision, Curvas ROC e PR"),
    ("5. Explicabilidade",   "Importância das features (impureza de Gini)"),
    ("6. MLOps",             "Modelo versionado + metadados JSON em models/"),
]
for etapa, desc in etapas:
    print(f"  {etapa:<22} → {desc}")
print("=" * 65)

# Exibe as métricas finais como resumo
print(f"\n🎯 Métricas finais no conjunto de teste:")
print(f"   ROC-AUC           : {test_metrics['roc_auc']:.4f}")
print(f"   Average Precision : {test_metrics['average_precision']:.4f}")
print(f"   F1-Score          : {test_metrics['f1_score']:.4f}")